# 🐝 Bee Detection Model Training

Train a custom YOLO11n model for bee detection using your Roboflow dataset.

**Your Dataset:**
- Workspace: `beemaster`
- Project: `beepollen2-uxs5w`

**Training Time:** ~30-45 minutes on free Colab GPU

---

## Setup Instructions

1. **Enable GPU**: Runtime → Change runtime type → GPU → Save
2. **Run all cells**: Runtime → Run all
3. **Wait for training**: Go get coffee ☕
4. **Download models**: Files will auto-download when done

In [ ]:
# ============================================================
# 1. Install Dependencies
# ============================================================
print("📦 Installing Ultralytics YOLO and Roboflow...\n")
!pip install -q ultralytics roboflow
print("✓ Installation complete!\n")

In [ ]:
# ============================================================
# 2. Check GPU
# ============================================================
import torch
import os

print("🔍 Checking GPU availability...\n")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU available: {gpu_name}")
    print(f"  Memory: {gpu_memory:.1f} GB\n")
else:
    print("⚠️  No GPU detected! Change runtime to GPU:\n")
    print("   Runtime → Change runtime type → GPU → Save\n")

In [ ]:
# ============================================================
# 3. Download Dataset from Roboflow
# ============================================================
from roboflow import Roboflow

# Your Roboflow credentials
ROBOFLOW_API_KEY = "tbWvZhQTMNtDp6KIvnOM"
WORKSPACE = "beemaster"
PROJECT = "beepollen2-uxs5w"
VERSION = 1  # Adjust if you have a specific version

print("📥 Downloading dataset from Roboflow...\n")
print(f"  Workspace: {WORKSPACE}")
print(f"  Project: {PROJECT}")
print(f"  Version: {VERSION}\n")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

print(f"\n✓ Dataset downloaded to: {dataset.location}")
print(f"\nDataset structure:")
!ls -lh {dataset.location}

In [ ]:
# ============================================================
# 4. Inspect Dataset
# ============================================================
import yaml

# Load data.yaml to see classes
data_yaml_path = f"{dataset.location}/data.yaml"
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("📊 Dataset Information:\n")
print(f"  Classes: {data_config.get('names', [])}")
print(f"  Number of classes: {data_config.get('nc', 'Unknown')}")
print(f"  Train images: {data_config.get('train', 'Unknown')}")
print(f"  Validation images: {data_config.get('val', 'Unknown')}")
print(f"\ndata.yaml path: {data_yaml_path}")

In [ ]:
# ============================================================
# 5. Train YOLO11n Model
# ============================================================
from ultralytics import YOLO

print("="*60)
print("🚀 Starting YOLO11n Training")
print("="*60)
print("\nThis will take ~30-45 minutes on T4 GPU.")
print("Feel free to grab some coffee! ☕\n")

# Load pretrained YOLO11n model
model = YOLO('yolo11n.pt')

# Training configuration optimized for bee detection
results = model.train(
    data=data_yaml_path,
    epochs=100,              # Good balance for small datasets
    imgsz=640,              # Standard size, Hailo optimized
    batch=16,               # Good for T4 GPU
    device=0,               # Use GPU
    patience=20,            # Early stopping if no improvement
    save=True,
    project='bee-detection',
    name='yolo11n-bee',
    
    # Optimization for small objects (bees)
    close_mosaic=10,        # Disable mosaic in last 10 epochs
    amp=True,               # Automatic mixed precision (faster)
    
    # Data augmentation tuned for bee detection
    hsv_h=0.015,           # Slight hue variation
    hsv_s=0.7,             # Saturation
    hsv_v=0.4,             # Value/brightness
    degrees=10.0,          # Slight rotation
    translate=0.1,         # Translation
    scale=0.5,             # Scale variation
    fliplr=0.5,            # Horizontal flip (OK for bees)
    flipud=0.0,            # No vertical flip (bees upright)
    mosaic=1.0,            # Mosaic augmentation
)

print("\n" + "="*60)
print("✓ Training Complete!")
print("="*60)

In [ ]:
# ============================================================
# 6. Validate Model
# ============================================================
print("\n📊 Validating model on test set...\n")

metrics = model.val()

print("\n" + "="*60)
print("📈 Validation Results")
print("="*60)
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

if metrics.box.map50 > 0.7:
    print("\n✓ Great performance! Model is ready for deployment.")
elif metrics.box.map50 > 0.5:
    print("\n⚠️  Decent performance. Consider more training data or epochs.")
else:
    print("\n⚠️  Low performance. May need more data or hyperparameter tuning.")

In [ ]:
# ============================================================
# 7. Export to ONNX (for Hailo)
# ============================================================
print("\n📦 Exporting to ONNX format (for Hailo conversion)...\n")

onnx_path = model.export(
    format='onnx',
    imgsz=640,
    simplify=True,         # Simplify for Hailo
    dynamic=False,         # Static shapes (required for Hailo)
    opset=11,             # ONNX opset version
)

print(f"\n✓ ONNX model exported to: {onnx_path}")

In [ ]:
# ============================================================
# 8. View Training Results
# ============================================================
from IPython.display import Image, display
import os

print("\n📊 Training Results Visualization\n")

results_dir = 'bee-detection/yolo11n-bee'

# Display training curves
results_img = f"{results_dir}/results.png"
if os.path.exists(results_img):
    print("Training Metrics:")
    display(Image(filename=results_img, width=800))

# Display confusion matrix
confusion_img = f"{results_dir}/confusion_matrix.png"
if os.path.exists(confusion_img):
    print("\nConfusion Matrix:")
    display(Image(filename=confusion_img, width=600))

# Display sample predictions
val_batch_img = f"{results_dir}/val_batch0_pred.jpg"
if os.path.exists(val_batch_img):
    print("\nSample Predictions:")
    display(Image(filename=val_batch_img, width=800))

In [ ]:
# ============================================================
# 9. Test on Sample Image
# ============================================================
import glob

print("\n🧪 Testing model on sample image...\n")

# Get a random validation image
val_images = glob.glob(f"{dataset.location}/valid/images/*")
if val_images:
    test_image = val_images[0]
    print(f"Test image: {test_image}\n")
    
    # Run prediction
    results = model.predict(test_image, conf=0.25, save=True)
    
    # Display result
    pred_img = f"{results_dir}/predict/image0.jpg"
    if os.path.exists(pred_img):
        print("\nPrediction Result:")
        display(Image(filename=pred_img, width=600))
    
    # Print detections
    for r in results:
        boxes = r.boxes
        print(f"\nDetected {len(boxes)} object(s):")
        for box in boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            print(f"  - Class: {model.names[cls]}, Confidence: {conf:.2f}")
else:
    print("No validation images found.")

In [ ]:
# ============================================================
# 10. Download Trained Models
# ============================================================
from google.colab import files
import shutil
import os

print("\n📥 Preparing models for download...\n")

# Create download directory
download_dir = 'trained_models'
os.makedirs(download_dir, exist_ok=True)

# Copy important files
weights_dir = f"{results_dir}/weights"

files_to_download = [
    (f"{weights_dir}/best.pt", f"{download_dir}/yolo11n_bee_best.pt"),
    (f"{weights_dir}/best.onnx", f"{download_dir}/yolo11n_bee_best.onnx"),
    (f"{results_dir}/results.png", f"{download_dir}/training_results.png"),
    (f"{results_dir}/confusion_matrix.png", f"{download_dir}/confusion_matrix.png"),
]

for src, dst in files_to_download:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"✓ Prepared: {dst}")

# Create labels.json for deployment
labels_json = {
    str(i): name for i, name in enumerate(model.names.values())
}

import json
with open(f"{download_dir}/labels_bee.json", 'w') as f:
    json.dump(labels_json, f, indent=2)
print(f"✓ Created: labels_bee.json")

# Create README with deployment instructions
readme = f"""# Bee Detection Model - Training Results

## Model Performance
- mAP50: {metrics.box.map50:.3f}
- mAP50-95: {metrics.box.map:.3f}
- Precision: {metrics.box.mp:.3f}
- Recall: {metrics.box.mr:.3f}

## Files
- `yolo11n_bee_best.pt` - PyTorch model (for testing)
- `yolo11n_bee_best.onnx` - ONNX model (for Hailo conversion)
- `labels_bee.json` - Class labels
- `training_results.png` - Training metrics
- `confusion_matrix.png` - Confusion matrix

## Deployment to Raspberry Pi

1. Convert ONNX to HEF (on your Mac):
   ```bash
   # Edit convert_onnx_to_HEF.sh to use yolo11n_bee_best.onnx
   ./api/models/convert_onnx_to_HEF.sh
   ```

2. Deploy to Pi:
   ```bash
   scp yolo11n_bee.hef labels_bee.json rpi:/tmp/
   ssh rpi "sudo mv /tmp/yolo11n_bee.* /opt/bee-monitoring/src/api/models/"
   ```

3. Test:
   ```bash
   curl "http://192.168.68.66/api/bee/ai/detect?stream_url=/opt/bee-monitoring/videos/your_bee_video.mov&ai_backend=hailo&annotate=1" -o result.jpg
   ```

## Classes Detected
{json.dumps(labels_json, indent=2)}
"""

with open(f"{download_dir}/README.md", 'w') as f:
    f.write(readme)
print(f"✓ Created: README.md")

# Zip everything
print("\n📦 Creating zip file...")
shutil.make_archive('bee_detection_model', 'zip', download_dir)

print("\n" + "="*60)
print("✓ ALL DONE! Downloading models...")
print("="*60)
print("\nDownloading bee_detection_model.zip...\n")

files.download('bee_detection_model.zip')

print("\n✓ Download complete!")
print("\nNext steps:")
print("  1. Extract the zip file on your Mac")
print("  2. Convert ONNX to HEF using convert_onnx_to_HEF.sh")
print("  3. Deploy to Raspberry Pi")
print("  4. Test with your bee video!")
print("\nSee README.md in the zip for detailed instructions.")

---

## 🎉 Training Complete!

Your trained bee detection model has been downloaded as `bee_detection_model.zip`.

### What's Next?

1. **Extract the zip file** on your Mac
2. **Convert to HEF** (Hailo format) using Docker
3. **Deploy to Raspberry Pi**
4. **Test with your bee video**

See the README.md file in the zip for complete deployment instructions!

---

### Training Summary

- ✅ Dataset downloaded from Roboflow
- ✅ YOLO11n trained on bee detection
- ✅ Model validated and exported to ONNX
- ✅ Ready for Hailo conversion and deployment

**Happy bee monitoring! 🐝**